# 6B Install ComfyUI in an SSH Session

**Purpose:** Manually install ComfyUI on Klique-managed compute, generate one sample image, save the result, and clean up.

**What we're doing**

Use the SSH Session from [Section 6](06_Running_an_SSH_Session.ipynb) as a development environment. Install upstream ComfyUI into its own Python environment and run a small SD 1.5 workflow.

**Why we're doing it**

Sections 1–4 ran a model through an app Klique packages for you. Most real workloads have no such app. ComfyUI is a good test of that case: it needs a GPU, a specific PyTorch build for that GPU, a multi-gigabyte model, and a long-running web server, all set up by hand. Running it on a plain SSH Session shows that Klique's scheduling, GPU allocation, and persistent workspaces apply to software you install yourself, not just to the apps it ships.

This makes the pieces visible: compute, Python dependencies, model weights, a web server, and a client request. This lab does not assume a prebuilt Klique ComfyUI app exists.

> **Where this lab was first tested.** I first ran this lab on an NVIDIA DGX Spark. That was a deliberate choice, not a convenience.
>
> Most GPU tutorials quietly assume an x86_64 machine with a discrete NVIDIA card. The DGX Spark breaks almost every one of those assumptions. Its GB10 chip pairs an ARM64 CPU with a Blackwell GPU. Its driver reports CUDA 13.0. The CPU and GPU share a single 128 GB pool of memory. And the image I launched shipped Ubuntu 22.04 with Python 3.10, older than ComfyUI recommends.
>
> Each of those is a place where a copy-and-paste install breaks. Install commands written for x86_64 do not apply. A plain `pip install torch` can quietly land a CPU-only build. `nvidia-smi` reports GPU memory as *Not Supported*. The `venv` module was missing entirely.
>
> If the lab works there, a conventional x86_64 NVIDIA server is the easier case. That is why 6B.1 and 6B.3 call out ARM64 and GB10 by name: those notes are the failure modes I hit, and the fixes that worked.

**Output:** One 512 × 512 PNG, recorded software/model versions, and a stopped session.

## 6B.1 Prerequisites

* Complete the SSH introduction. Use a Linux session with Bash, Git, curl, Python with venv support, and an approved GPU-capable container.
* Suggested lab allocation: one supported GPU with **8 GB VRAM**, **16 GB system RAM**, and **25 GB free workspace storage**. These are conservative planning targets for this one-image exercise, not vendor minimums or measured guarantees.
* Confirm the remote GPU model, driver/runtime and container device access with the administrator. The GPU in your laptop does not determine the remote runtime.
* Have permission to download packages and the approximately 2.13 GB checkpoint, or use approved mirrors. Keep source, models, outputs, and environment under your persistent workspace.
* Confirm SSH TCP forwarding is permitted if connecting a client from your laptop. The [Klique SSH app documentation](https://klique.ai/docs/latest/docs/webapp/applications/apps_ssh_session) supplies instance connection details but does not guarantee forwarding policy or network namespace layout. Do not open a public port as a workaround.

### Choose the GPU runtime deliberately

| Remote hardware | Required match |
|---|---|
| NVIDIA | CUDA-enabled PyTorch compatible with GPU architecture and host driver |
| AMD | ROCm-enabled PyTorch on a supported GPU/OS/runtime combination |
| CPU only | Possible for diagnosis, but much slower; not the recommended image-generation exercise |

Consult [ComfyUI system requirements](https://docs.comfy.org/installation/system_requirements), [PyTorch's installation selector](https://pytorch.org/get-started/locally/), and, for AMD, [AMD's PyTorch compatibility/install guidance](https://rocm.docs.amd.com/projects/ai-ecosystem/en/latest/frameworks/pytorch/install.html). Use a Python version accepted by both the chosen wheels and ComfyUI; Python 3.12 or 3.13 is a practical starting point. Do not install host drivers or change device permissions from this lab.

> **Note: ARM64 hosts such as the NVIDIA DGX Spark (GB10).** Run `uname -m` in the remote shell. If it prints `aarch64`, the session is ARM64, and an x86_64 PyTorch command from the selector will not apply. Pick the Linux ARM64 build that matches the CUDA version `nvidia-smi` reports, and confirm `torch.cuda.is_available()` is `True` in 6B.3 before downloading the model. This note is general PyTorch guidance, not something the Klique SSH Session page documents; ask your administrator which image is approved for the queue.

> **Note: NVIDIA GB10 (DGX Spark).** On a GB10 session, `uname -m` prints `aarch64` and `nvidia-smi` reports a driver that supports **CUDA 13.0**. Install the CUDA 13.0 ARM64 build from PyTorch's own index, inside the activated venv in 6B.3:
>
> ```bash
> python3 -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
> ```
>
> `--index-url` keeps pip on PyTorch's CUDA index; without it, pip can install a CPU-only build from PyPI, and the 6B.3 GPU check fails. In November 2026 that index had ARM64 builds for Python 3.10 as well as 3.12 (torch 2.14.1), so PyTorch itself does not require relaunching a GB10 image that ships Python 3.10.
>
> GB10 shares one pool of memory between the CPU and the GPU. `nvidia-smi` shows **Not Supported** under memory usage, and the 6B.3 check's "VRAM GiB" line reports most of the system's memory rather than a separate GPU size. Both are expected.

> **Python version.** Check `python3 --version` in the remote shell. This lab recommends Python 3.12 or 3.13 (3.11 at minimum), and some images ship an older one (for example 3.10). If yours is older, stop the instance and launch a new one with a different **Container → Image**, then set **Restore Interactive Workspace ID** to the stopped session's ID. Klique's [launch form customization page](https://klique.ai/docs/latest/docs/deploying_klique/app_launch_form_custom) shows official Python images such as `python:3.12` and `python:3.11` as image choices. Klique's SSH Session page only requires that the image meet the app's **Container requirements**, so confirm with your administrator that the image you pick is approved for the queue and GPU. The docs do not describe installing a newer Python from the launch form.
>
> Python 3.10 is not always a blocker. On a GB10 session with Python 3.10.12, PyTorch 2.14.1 (CUDA 13.0) and ComfyUI's `requirements.txt` both installed, and the 6B.3 GPU check passed. Before relaunching, try the install: relaunch only if a PyTorch wheel or a ComfyUI requirement rejects your Python version.

> **Checkpoint:** Record the approved queue, image, GPU, Python version, and compatible PyTorch installation command.

## 6B.2 Prepare the Remote Workspace

Use the SSH introduction to launch and connect. Set a persistent path such as `~/workspace` and an idle timeout appropriate for setup and generation; record the session ID. Commands below run in the **remote Linux shell** unless labeled otherwise.

Also record the CPU architecture with `uname -m` (see the ARM64 note in 6B.1).

```bash
apt-get update && apt-get install -y curl python3-venv
hostname
uname -m
python3 --version
git --version
curl --version
mkdir -p ~/workspace/comfyui-lab
cd ~/workspace/comfyui-lab
df -h .
```

> **Why the first line installs packages.** Some images ship without `curl`, or without Python's `venv` module (`python3 -m venv` then fails in 6B.3 with `ensurepip is not available`). The SSH session runs as `root`, so the block installs both before checking anything. If they are already present, the line changes nothing.
>
> System packages live outside your persistent workspace. A restored session brings back `~/workspace`, including the venv and the model, but not packages installed with `apt-get`. Run the first line again after every restore.

Substitute your configured workspace path throughout if different. Keep this installation outside KliqueNotebooks so model binaries and generated files cannot accidentally enter that repository.

For NVIDIA, inspect `nvidia-smi` if provided by the approved image. For AMD, use the administrator's approved device diagnostic. Missing tools or devices are reasons to check the image and assignment, not to guess privileged Docker flags.

> **Screenshot placeholder:** SSH instance ready with the approved queue/image and persistence settings; paired remote terminal showing Python version and GPU diagnostics. Mask connection credentials.

<!-- screenshot: images/comfyui_environment.png (Ready SSH instance and remote Python/GPU diagnostics; credentials masked.) -->

## 6B.3 Install ComfyUI and Dependencies

The [manual installation guide](https://docs.comfy.org/installation/manual_install) separates environment creation, source checkout, GPU dependencies, and ComfyUI requirements. This lab uses Python venv for isolation.

In the remote shell:

```bash
cd ~/workspace/comfyui-lab
python3 -m venv .venv
source .venv/bin/activate
python3 -m pip install --upgrade pip
git clone https://github.com/Comfy-Org/ComfyUI.git
cd ComfyUI
git rev-parse HEAD
```

> **If `python3 -m venv .venv` fails with `ensurepip is not available`:** the image's Python is missing its venv package, which is common on Ubuntu 22.04 images (Python 3.10). The session runs as `root`, so install the package, remove the half-built environment, and create it again:
>
> ```bash
> apt-get update && apt-get install -y python3-venv
> rm -rf .venv
> python3 -m venv .venv
> source .venv/bin/activate
> ```
>
> Then continue from `python3 -m pip install --upgrade pip`.

If the directory already exists, inspect it instead of cloning or resetting over it. Record the commit so your sample is reproducible.

**Now install PyTorch using the compatible command selected in 6B.1**, inside the activated environment. Use `python3 -m pip` in place of the selector's `pip`/`pip3` executable so installation targets this interpreter. Choose one GPU backend; do not combine CUDA and ROCm wheels. A single hard-coded wheel version cannot cover every queue.

Then:

```bash
python3 -m pip install -r requirements.txt
python3 -m pip check
python3 -m pip freeze > ../requirements-observed.txt
python3 - <<'PY'
import sys
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("ROCm build:", torch.version.hip)
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("GPU not visible: resolve runtime/device compatibility before generating.")
print("Device:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
PY
```

> **Note: `pip check` on ARM64.** On ARM64 sessions such as GB10, `pip check` can print `nvidia-cusparselt-cu13 … is not supported on this platform`. PyTorch's ARM64 build lists cuSPARSELt as a dependency, but that package's metadata does not declare ARM64 support. cuSPARSELt only speeds up sparse matrix math, which this lab's SD 1.5 workflow does not use. If the GPU check below it passes, ignore the warning. Treat any other `pip check` message as a real conflict.

PyTorch uses the `torch.cuda` device interface for ROCm too; the HIP build value identifies that backend. A recorded package list helps reconstruction but is not proof of portability to another image.

> **Checkpoint:** Dependency checks pass and the intended remote GPU is visible.

> **Screenshot placeholder:** Remote terminal showing the ComfyUI commit, successful pip check, PyTorch backend, device name, and VRAM. Omit package-index credentials.

<!-- screenshot: images/comfyui_dependencies.png (Successful dependency and GPU verification in the isolated environment.) -->

## 6B.4 Download the Complete Model for This Workflow

Use the SD 1.5 FP16 checkpoint linked by ComfyUI's basic tutorial. This is the **only model file** needed for this sample: its checkpoint loader supplies the diffusion model, CLIP text encoder, and VAE. No separate VAE, LoRA, ControlNet, upscaler, or custom node is required.

| Asset | Verified source and destination |
|---|---|
| `v1-5-pruned-emaonly-fp16.safetensors` | [Comfy-Org file page](https://huggingface.co/Comfy-Org/stable-diffusion-1.5/blob/main/v1-5-pruned-emaonly-fp16.safetensors); save in `ComfyUI/models/checkpoints/` |
| Model license | [Model card](https://huggingface.co/Comfy-Org/stable-diffusion-1.5) lists **CreativeML OpenRAIL-M**; review the [license text](https://huggingface.co/spaces/CompVis/stable-diffusion-license) and its use restrictions before downloading |
| ComfyUI code | Governed by the LICENSE in the cloned upstream repository, separately from the model license |

The repository includes a setup script, [`setup_comfyui_lab.py`](setup_comfyui_lab.py), that downloads the checkpoint, verifies its SHA-256, and moves it into place. It also writes the 6B.7 workflow for later. It uses only the Python standard library and is safe to rerun: a checkpoint that is already present and verified is not downloaded again. Run it in the remote shell, from the folder that holds `ComfyUI/`:

```bash
cd ~/workspace/comfyui-lab
python3 ~/workspace/KliqueNotebooks/06_SSH/setup_comfyui_lab.py
```

The path assumes you cloned KliqueNotebooks into `~/workspace` in Section 6.4. If you did not, clone it now with `git clone https://github.com/FrankLaVigne/KliqueNotebooks.git ~/workspace/KliqueNotebooks`. If you cloned it earlier, run `git -C ~/workspace/KliqueNotebooks pull` to get the current lab files. Read the script before running it; it is short.

Expect a `✅ Verified:` line for the checkpoint, then two `✅` lines for the workflow files.

The venv does not need to be active, because the script uses only the Python standard library. If ComfyUI is already running when you run the script, reload the browser page so the new workflow appears under **Workflows**.

**Manual alternative.** These commands do the same download and check by hand. Run them from the remote `ComfyUI` directory:

```bash
curl --fail --location --retry 3 --output models/checkpoints/v1-5-pruned-emaonly-fp16.safetensors.part https://huggingface.co/Comfy-Org/stable-diffusion-1.5/resolve/main/v1-5-pruned-emaonly-fp16.safetensors
python3 - <<'PY'
from pathlib import Path
import hashlib

part = Path("models/checkpoints/v1-5-pruned-emaonly-fp16.safetensors.part")
expected = "e9476a13728cd75d8279f6ec8bad753a66a1957ca375a1464dc63b37db6e3916"
with part.open("rb") as stream:
    digest = hashlib.sha256()
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(block)
    actual = digest.hexdigest()
if actual != expected:
    raise SystemExit(f"Checksum mismatch: {actual}; do not load this file.")
destination = part.with_suffix("")
if destination.exists():
    raise SystemExit("Checkpoint already exists; inspect it before replacing anything.")
part.rename(destination)
print("Verified:", destination, destination.stat().st_size, "bytes")
PY
```

The expected SHA-256 is published on the linked file page. A changed upstream file needs a new source review, not removal of the check. If the verified file is already present, skip the download. Do not rename an HTML error page or a Git pointer into a checkpoint.

> **Checkpoint:** The checksum passes and the final filename is in `models/checkpoints`.

> **Screenshot placeholder:** Checksum success and checkpoint filename/size in the remote terminal; no model data needs to be displayed.

<!-- screenshot: images/comfyui_checkpoint.png (Verified SHA-256 and checkpoint destination.) -->

## 6B.5 Start the Server on Remote Loopback

In the remote shell, with the environment activated and working directory `ComfyUI`:

```bash
python3 main.py --listen 127.0.0.1 --port 8188 --disable-auto-launch
```

Leave this foreground process running in **remote terminal A**. The [upstream CLI definition](https://github.com/Comfy-Org/ComfyUI/blob/master/comfy/cli_args.py) documents these flags. Supplying `--listen` without an address can bind all interfaces; keep the explicit loopback address.

Open **remote terminal B** using the same instance's connection command and check:

```bash
curl --fail http://127.0.0.1:8188/system_stats
```

Expect JSON describing the server environment. If it fails, inspect terminal A first.

> **Checkpoint:** ComfyUI is running and responds inside the SSH session. This is a self-hosted service; do not assume it has public authentication or gateway protection.

> **Screenshot placeholder:** Terminal A showing the explicit loopback launch command and ready log, beside terminal B's successful system_stats response.

<!-- screenshot: images/comfyui_server.png (Loopback server ready and remote API health response.) -->

## 6B.6 Reach It Through SSH

Keep the server terminal open. In a **second local terminal**, take the instance's working SSH connection command and add the forwarding options below **before its destination**. Preserve its actual port, identity, proxy/jump options, and authentication method. For a plain OpenSSH command, this is the shape, not a literal command:

```text
ssh -N -T -o ExitOnForwardFailure=yes -L 127.0.0.1:8188:127.0.0.1:8188 -p ACTUAL_SSH_PORT ACTUAL_USER@ACTUAL_SSH_HOST
```

The [OpenSSH manual](https://man.openbsd.org/ssh.1) documents local forwarding: the first address is your laptop's listener; the second is reached from the remote SSH server. `-N` runs no remote command. Authenticate using the instance instructions and verify host keys normally. Leave this terminal running.

This requires the SSH server to permit TCP forwarding and share/reach ComfyUI's loopback network context. Klique's published SSH page does not establish those facts for every deployment. If the provided connection uses a wrapper or forwarding fails, have the administrator confirm the supported command and topology. Do not change SSH policy, expose port 8188 publicly, or bind ComfyUI to all interfaces to bypass a failure.

Open **http://127.0.0.1:8188** in your laptop browser. For a local port conflict, use `-L 127.0.0.1:18188:127.0.0.1:8188`, then use port `18188` in the browser and API client. The remote server still uses 8188.

> **Checkpoint:** The ComfyUI canvas loads through the tunnel. `ExitOnForwardFailure` catches tunnel setup problems; it does not prove the remote application is reachable.

> **Screenshot placeholder:** Local browser at the loopback URL displaying ComfyUI, alongside the tunnel command with host/user details redacted.

<!-- screenshot: images/comfyui_tunnel.png (Localhost browser canvas reached through authenticated SSH forwarding.) -->

## 6B.7 Generate and Save the Browser Sample

On your laptop, download the [official basic workflow PNG](https://raw.githubusercontent.com/Comfy-Org/example_workflows/main/text-to-image-workflow.png) linked by the [ComfyUI text-to-image tutorial](https://docs.comfy.org/tutorials/basic/text-to-image). Drag the original PNG onto the canvas to load its embedded workflow; a screenshot copy loses that metadata.

Use the built-in nodes only. Select `v1-5-pruned-emaonly-fp16.safetensors` in **Load Checkpoint**. Set these lab values:

| Node/input | Value |
|---|---|
| Positive CLIP text | `a small red cabin beside a calm lake, pine trees, morning sunlight, watercolor illustration` |
| Negative CLIP text | `text, watermark, blurry` |
| Empty Latent Image | width 512, height 512, batch size 1 |
| KSampler | seed 42; control_after_generate fixed; steps 20; cfg 7; sampler euler; scheduler normal; denoise 1 |
| Save Image filename_prefix | `klique_ui` |

Inspect the connections: checkpoint MODEL/CLIP/VAE feed sampling, text encoding, and decoding; decoded pixels feed Save Image. Click **Queue** or press **Ctrl+Enter** once.

The image should appear in **Save Image** and be written under remote `ComfyUI/output/`. Right-click the resulting image to save a local copy. Confirm its dimensions are 512 × 512.

> **Shortcut:** the 6B.4 setup script saved this same workflow, with the values above, as **klique_lab_sd15**. Open the **Workflows** panel in ComfyUI's left sidebar and click it. Build the workflow yourself first if you can; then open `klique_lab_sd15` to compare it with yours.

> **Checkpoint:** One image is visible and saved locally and remotely. Artistic quality is secondary; this proves the model, workflow, and server work together.

> **Screenshot placeholder:** Entire seven-node workflow with chosen checkpoint, 512 × 512 settings and resulting cabin image; include the loopback address.

<!-- screenshot: images/comfyui_ui_sample.png (Completed SD1.5 workflow and saved 512x512 cabin image.) -->

## 6B.8 Export the Same Workflow for the API

In the browser, use **File → Export Workflow (API)** and save the download as `workflow_api.json` on your laptop. Keep a normal canvas save separately if you want its layout.

| JSON type | What it contains | Use |
|---|---|---|
| Canvas/save workflow | Editor graph plus layout metadata | Reopen and edit visually |
| API workflow | Node-ID map with `class_type` and `inputs` | Submit inside the request's `prompt` field |

A connection such as `["4", 0]` identifies a node output, not an editor position. Do not POST the normal canvas JSON or the workflow PNG. See [Workflow API Format](https://docs.comfy.org/development/api-development/workflow-api-format).

Open the exported file and confirm the checkpoint name and SaveImage node. For this sample, every node runs locally; no Comfy Cloud key or paid API node is needed.

The 6B.4 setup script also wrote an API-format copy, `~/workspace/comfyui-lab/workflow_api.json`, on the remote machine. Compare it with your export: the node IDs and link pairs such as `["4", 1]` should match, and only values you changed in the browser should differ. The repository has the same file as [`06_SSH/workflow_api.json`](workflow_api.json), and 6B.9 uses that copy. Keep your browser export on your laptop as a reference.

> **Screenshot placeholder:** Export Workflow (API) menu and the downloaded JSON showing class_type/inputs, without opening unrelated local files.

<!-- screenshot: images/comfyui_api_export.png (API export menu and node-map JSON.) -->

## 6B.9 Submit, Poll, and Download with Python

The client script, [`run_comfy_sample.py`](run_comfy_sample.py), sends the lab workflow to ComfyUI's API, waits for the image, and saves it. It runs in the session, next to the server, so it does not use the SSH tunnel. It uses only the Python standard library, so the venv does not need to be active.

The script and the workflow it sends, [`workflow_api.json`](workflow_api.json), are both in your Section 6.4 clone of this repository. Keep ComfyUI running in **remote terminal A**. In **remote terminal B**, go to the clone's `06_SSH` folder and get the latest files:

```bash
cd ~/workspace/KliqueNotebooks/06_SSH
git pull
```

Then run the script:

```bash
python3 run_comfy_sample.py
```

The script asks for an image prompt. Press **Enter** to accept the default, a red barn in a watercolor meadow, or type your own. To run it again with a different prompt, pass the prompt in quotes:

```bash
python3 run_comfy_sample.py "a lighthouse on a rocky coast at dusk, oil painting"
```

Only the positive prompt changes. The negative prompt, 512 × 512 size, and sampler settings come from `workflow_api.json`. The script sets the seed to 43 and the filename prefix to `klique_api`, so this image is distinct from the browser sample. To send your own browser export instead, put it in the folder you run the script from; a `workflow_api.json` in the current folder takes precedence over the repository copy.

This is lab-authored HTTP polling code using the documented [server routes](https://docs.comfy.org/development/comfyui-server/comms_routes) and [image retrieval protocol](https://github.com/Comfy-Org/ComfyUI/blob/master/script_examples/websockets_api_example.py). The upstream examples also offer WebSocket progress updates. The client submits once, waits at most ten minutes, and downloads only completed output images. The full code is below, so you can read what it does before running it:

```python
"""Send the 6B lab workflow to ComfyUI, wait for it, and save the image.

Run it in the session while ComfyUI is running:

    cd ~/workspace/KliqueNotebooks/06_SSH
    python3 run_comfy_sample.py                      # asks for a prompt
    python3 run_comfy_sample.py "a blue sailboat"    # uses this prompt

It reads workflow_api.json from the current folder if there is one (for example,
your own export); otherwise it uses the copy next to this script. The image and
receipt are saved in the current folder. Uses only the Python standard library.
"""
import json
import sys
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import urlencode, quote
from urllib.request import Request, urlopen

BASE = "http://127.0.0.1:8188"
DEFAULT_PROMPT = "a red barn in a green meadow, rolling hills, afternoon sunlight, watercolor illustration"

workflow_path = Path("workflow_api.json")
if not workflow_path.exists():
    workflow_path = Path(__file__).resolve().parent / "workflow_api.json"
if not workflow_path.exists():
    raise SystemExit("No workflow_api.json here or next to this script.")
print("Workflow:", workflow_path.resolve())
workflow = json.loads(workflow_path.read_text(encoding="utf-8"))
if not isinstance(workflow, dict) or not workflow or not all(
    isinstance(node, dict) and "class_type" in node and "inputs" in node
    for node in workflow.values()
):
    raise SystemExit("Export API format, not the canvas save format.")

# Identify nodes by class rather than assuming the editor's numeric IDs.
samplers = [n for n in workflow.values() if n["class_type"] == "KSampler"]
savers = [n for n in workflow.values() if n["class_type"] == "SaveImage"]
if len(samplers) != 1 or len(savers) != 1:
    raise SystemExit("Use the single-sampler, single-SaveImage lab workflow.")

# The sampler's "positive" input links to the text node to replace, e.g. ["6", 0].
positive_id = samplers[0]["inputs"]["positive"][0]
positive = workflow.get(positive_id)
if positive is None or positive["class_type"] != "CLIPTextEncode":
    raise SystemExit("The sampler's positive input is not a CLIP text node.")

if len(sys.argv) > 1:
    text = " ".join(sys.argv[1:]).strip()
else:
    print("Image prompt (press Enter for the default):")
    print("  default:", DEFAULT_PROMPT)
    text = input("> ").strip()
text = text or DEFAULT_PROMPT
print("Prompt:", text, flush=True)

positive["inputs"]["text"] = text
samplers[0]["inputs"]["seed"] = 43
savers[0]["inputs"]["filename_prefix"] = "klique_api"

def fetch(path, payload=None):
    data = None if payload is None else json.dumps(payload).encode()
    headers = {} if data is None else {"Content-Type": "application/json"}
    request = Request(BASE + path, data=data, headers=headers)
    try:
        with urlopen(request, timeout=30) as response:
            return response.read()
    except HTTPError as error:
        raise RuntimeError(f"HTTP {error.code}: {error.read().decode(errors='replace')}") from error

def get_json(path, payload=None):
    return json.loads(fetch(path, payload))

# Submit once. Do not automatically retry POST after a network timeout.
receipt = get_json("/prompt", {"prompt": workflow})
if receipt.get("error") or receipt.get("node_errors"):
    raise SystemExit(json.dumps(receipt, indent=2))
prompt_id = receipt["prompt_id"]
print("Submitted:", prompt_id, flush=True)
Path("prompt-receipt.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")

deadline = time.monotonic() + 600
record = None
while time.monotonic() < deadline:
    record = get_json("/history/" + quote(prompt_id, safe="")).get(prompt_id)
    if record is not None:
        status = record.get("status", {})
        if status.get("status_str") == "error":
            raise SystemExit(json.dumps(status, indent=2))
        if status.get("completed"):
            break
    time.sleep(2)
else:
    raise SystemExit("Timed out waiting; job may still run. Inspect queue/server before resubmitting.")

images = [
    image
    for output in record.get("outputs", {}).values()
    for image in output.get("images", [])
    if image.get("type") == "output"
]
if not images:
    raise SystemExit("Finished without an output image; inspect history and SaveImage.")
for index, image in enumerate(images, 1):
    query = urlencode({key: image[key] for key in ("filename", "subfolder", "type")})
    png = fetch("/view?" + query)
    if not png.startswith(b"\x89PNG\r\n\x1a\n"):
        raise SystemExit("Server response is not a PNG.")
    # Fixed local naming avoids trusting server-supplied paths.
    destination = Path(f"api-{prompt_id}-{index}.png")
    with destination.open("xb") as stream:
        stream.write(png)
    print("Saved:", destination.resolve(), "bytes:", len(png))
```

Expected: the workflow path, your prompt, a prompt ID, and a saved path such as `~/workspace/KliqueNotebooks/06_SSH/api-<prompt id>-1.png`. Git ignores these output files. The server also keeps its own copy in `ComfyUI/output/` with the prefix `klique_api`. To confirm the image is 512 × 512 without leaving the terminal, read the size from its PNG header:

```bash
python3 -c "import struct, glob; f = sorted(glob.glob('api-*.png'))[-1]; print(f, struct.unpack('>II', open(f, 'rb').read(24)[16:24]))"
```

To copy the images to your laptop, open a **new local terminal** in the folder where you want them. Use the same port, user, and host as the 6B.6 tunnel command; note that `scp` takes the port as a capital `-P`:

```bash
scp -P ACTUAL_SSH_PORT 'ACTUAL_USER@ACTUAL_SSH_HOST:~/workspace/KliqueNotebooks/06_SSH/api-*.png' .
```

The quotes keep your local shell from expanding `~` and `*`, so the remote side matches every API image. The command works the same in macOS, Linux, and Windows PowerShell terminals. Open a copied PNG to view it.

Adjust `BASE` in the script only if you started ComfyUI on a port other than 8188. If the server reports a `ckpt_name` error, the 6B.4 checkpoint is missing; finish 6B.4 and try again.

If POST times out, the request may already be queued. Inspect the server and queue before submitting again. A polling timeout does not cancel generation.

> **Checkpoint:** Both browser and API paths generated an image. Keep the receipt with your software version notes; a fixed seed does not guarantee identical pixels across different runtimes.

> **Screenshot placeholder:** Remote terminal B showing the prompt, prompt ID, saved PNG path, and 512 × 512 size check.

<!-- screenshot: images/comfyui_api_sample.png (Successful submission, completion and retrieved PNG.) -->

## 6B.10 Save Work and Shut Down

1. Save the browser PNG and the API PNGs you copied with `scp` on your laptop. In `~/workspace/KliqueNotebooks/06_SSH`, keep `prompt-receipt.json`. Record the ComfyUI commit, checkpoint hash, Python/PyTorch versions, and chosen container image.
2. Wait for the job to finish. In **remote terminal A**, press **Ctrl+C** to stop ComfyUI and wait for the shell prompt.
3. In remote terminal B, repeat the `system_stats` request; it should fail because this server is stopped. If it still succeeds, identify the process before assuming cleanup is complete.
4. In the **local tunnel terminal**, press **Ctrl+C**. A fresh request to the local port should now fail; a cached canvas alone does not prove a server is still running.
5. Exit remote shells. In Klique, stop your SSH instance and verify stopped status.

Keep model weights and environment files only where your workspace/storage policy allows. Snapshots can be large. A restored workspace brings files into a fresh session; it does not restore a running web server or guarantee that compiled packages work with a different GPU image. Recheck the runtime, start ComfyUI again, and use the new instance's connection details.

No recursive deletion is needed to complete this lab.

> **Screenshot placeholder:** Returned shell prompt after Ctrl+C, failed health request, and stopped SSH instance dashboard.

<!-- screenshot: images/comfyui_cleanup.png (Server stopped, tunnel closed, and Klique instance stopped.) -->

## Troubleshooting

| Symptom | What to check |
|---|---|
| Missing wheel / incompatible package | Match Python, GPU architecture, host driver/runtime and selected PyTorch build. Consult the linked compatibility docs. |
| GPU unavailable | Confirm assignment and device access in the approved image; do not install host drivers here. |
| Out of memory | Keep batch size 1 and 512 × 512; remove competing workloads. Consider a compatible larger queue. |
| Checksum mismatch | Stop. Check download completion and the source's published hash; do not load the partial file. |
| Checkpoint not listed | Confirm exact filename and models/checkpoints location; refresh model choices or restart the server. |
| Browser fails but remote health check works | Inspect tunnel port, SSH forwarding policy, proxy options and network context. |
| Forwarding administratively prohibited | Ask the administrator for a supported authenticated route; do not publish the unauthenticated service. |
| API rejects class_type/inputs | Export API format rather than the canvas save format. |
| API validation node_errors | Read the response for model filename, node or input errors; use the basic built-in workflow. |
| API history reports error | Inspect the status and server log; do not treat queue acceptance as generation success. |
| `No such file or directory: 'workflow_api.json'` | Your clone has an older client. Run `git pull` in `~/workspace/KliqueNotebooks/06_SSH`, then rerun the script there. |
| Polling timeout | Check the recorded prompt ID and queue/server before retrying; the job may still run. |
| Missing files after restore | Check persistent path/session ID and the storage policy; local downloads remain separate. |

## Wrap-Up

You have manually installed the application, verified the complete model, run a local-only server, and used both its UI and API to save an image.

**Back:** [SSH introduction](06_Running_an_SSH_Session.ipynb)  
**Next:** [Running VS Code](../07_VSCode/07_Running_VS_Code.ipynb)  
**Index:** [README](../README.md)